# Explore: Inflation Forecasting with Prophet

This notebook walks you through the inflation agent's logic step by step.
Run each cell and observe the output. Modify values to experiment!

**What you'll learn:**
1. How to call the World Bank API
2. How to prepare data for Prophet
3. How Prophet fits a model and forecasts
4. How to read confidence intervals

In [ ]:
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Suppress Prophet's verbose logging
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

print('All imports successful!')

## Step 1: Fetch CPI Data from the World Bank API

The World Bank provides free economic data for every country.
We're fetching Singapore's CPI inflation rate (annual %).

In [ ]:
# Fetch Singapore CPI data (2015-2024)
url = 'https://api.worldbank.org/v2/country/SGP/indicator/FP.CPI.TOTL.ZG'
params = {'date': '2015:2024', 'format': 'json', 'per_page': 100}

response = requests.get(url, params=params)
print(f'Status: {response.status_code}')

data = response.json()
print(f'Records returned: {len(data[1]) if len(data) > 1 else 0}')

# Parse into a clean DataFrame
records = []
for entry in data[1]:
    if entry['value'] is not None:
        records.append({'year': int(entry['date']), 'cpi': round(entry['value'], 2)})

df = pd.DataFrame(records).sort_values('year').reset_index(drop=True)
print('\nSingapore CPI Inflation (%):')
df

## Step 2: Visualize the Historical Data

Always plot your data before modelling. Look for:
- Trends (going up or down?)
- Outliers (COVID-19 in 2020?)
- Patterns (seasonal?)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df['year'], df['cpi'], color=['red' if v < 0 else 'steelblue' for v in df['cpi']])
plt.axhline(y=0, color='black', linewidth=0.5)
plt.xlabel('Year')
plt.ylabel('CPI Inflation (%)')
plt.title('Singapore CPI Inflation Rate (2015-2024)')
plt.xticks(df['year'])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAverage inflation: {df['cpi'].mean():.2f}%")
print(f"Peak: {df['cpi'].max():.2f}% ({df.loc[df['cpi'].idxmax(), 'year']})")
print(f"Lowest: {df['cpi'].min():.2f}% ({df.loc[df['cpi'].idxmin(), 'year']})")

## Step 3: Prepare Data for Prophet

Prophet requires exactly two columns:
- `ds`: a datetime column
- `y`: the value to forecast

Our data is annual, so we expand each year into 12 monthly points.
This gives Prophet enough data points to detect trends.

In [ ]:
# Expand annual data to monthly
monthly_dates = []
monthly_values = []

for _, row in df.iterrows():
    for month in range(1, 13):
        monthly_dates.append(pd.Timestamp(year=int(row['year']), month=month, day=1))
        # Add tiny noise so Prophet doesn't see flat segments
        noise = np.random.normal(0, 0.05)
        monthly_values.append(row['cpi'] + noise)

prophet_df = pd.DataFrame({'ds': monthly_dates, 'y': monthly_values})

print(f'Data points: {len(prophet_df)}')
print(f'Date range: {prophet_df["ds"].min()} to {prophet_df["ds"].max()}')
prophet_df.head()

## Step 4: Fit Prophet Model and Forecast

Prophet decomposes the time series into:
- **Trend**: overall direction
- **Yearly seasonality**: repeating annual pattern
- **Residual**: unexplained noise

The `interval_width=0.95` means 95% of future values should fall
within the predicted bounds.

In [ ]:
from prophet import Prophet

# Create and fit the model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,   # No weekly pattern in annual data
    daily_seasonality=False,    # No daily pattern
    interval_width=0.95,        # 95% confidence interval
)

model.fit(prophet_df)

# Forecast 12 months ahead
future = model.make_future_dataframe(periods=12, freq='MS')
forecast = model.predict(future)

# Show just the forecasted period
forecast_only = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)
forecast_only.columns = ['Date', 'Predicted', 'Lower Bound', 'Upper Bound']
forecast_only.round(2)

## Step 5: Visualize the Forecast

The shaded area is the confidence interval — wider = more uncertainty.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Historical data
ax.plot(prophet_df['ds'], prophet_df['y'], 'b.', alpha=0.3, label='Historical (monthly)')

# Forecast line
ax.plot(forecast['ds'], forecast['yhat'], 'r-', linewidth=2, label='Forecast')

# Confidence interval
ax.fill_between(
    forecast['ds'],
    forecast['yhat_lower'],
    forecast['yhat_upper'],
    alpha=0.2, color='red', label='95% Confidence'
)

# Mark the forecast start
forecast_start = prophet_df['ds'].max()
ax.axvline(x=forecast_start, color='green', linestyle='--', alpha=0.5, label='Forecast starts')

ax.set_xlabel('Date')
ax.set_ylabel('CPI Inflation (%)')
ax.set_title('Singapore CPI Inflation: Historical + Forecast')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Prophet's Component Breakdown

This shows how Prophet decomposed the data into trend + seasonality.

In [ ]:
fig = model.plot_components(forecast)
plt.tight_layout()
plt.show()

## Exercises for You

1. **Change the country**: Replace `SGP` with `MYS` (Malaysia) or `USA` and re-run
2. **Change forecast period**: Try 24 months. How much wider do the intervals get?
3. **Remove the noise**: Set `noise = 0` and see how Prophet handles flat segments
4. **Try a different indicator**: Use `NY.GDP.MKTP.KD.ZG` for GDP growth instead of CPI